In [1]:
# %%
import tskit
import pandas as pd
import allel
import numpy as np
import pickle
from collections import Counter
from collections import defaultdict
import matplotlib.pyplot as plt
import sys

chromosome=15

/tmp/ipykernel_2301175/1371703484.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# Read the tree sequence
chrTree = tskit.load("Masked_snps/Chr" + str(chromosome) + ".trees")
tables = chrTree.dump_tables()
chrTree.num_sites
chrTree.num_trees


# %%
# Read vcf
vcf = allel.read_vcf("../Data/Chr" + str(chromosome) + "_imputed_AllInfo.vcf.gz")
# Get positions and genotypes
pos = vcf['variants/POS']
gt = vcf['calldata/GT']
ref = vcf['variants/REF']
alt = vcf['variants/ALT'][:, 0]

/home/jobste/bin/anaconda3/envs/tsinfer/lib/python3.9/site-packages/allel/io/vcf_read.py:1732: UserWarning: invalid INFO header: '##INFO=<ID=MajorOutgroup,Number=1,Type=String,Description="Major allele in the outgroup>"\n'
  warnings.warn('invalid INFO header: %r' % header)


In [3]:
vcfPos = vcf['variants/POS']
treePos = [int(site.position) for site in chrTree.sites()]


In [19]:
oneTree = chrTree.at_index(12)
print(oneTree.index)    

intervalBoolean = (vcfPos > oneTree.interval.left) & (vcfPos < oneTree.interval.right)
intervalPositions = vcfPos[intervalBoolean]
missingBoolean = list(map(lambda x: x not in treePos, intervalPositions))
missingGenotypes = gt[intervalBoolean][:, :, 0][missingBoolean]
missingPositions = pos[intervalBoolean][missingBoolean]
missingAlleles = list(zip(ref[intervalBoolean][missingBoolean], alt[intervalBoolean][missingBoolean]))




12


In [18]:
missingGenotypes


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int8)

list(map(lambda x: x in treePos, vcfPos))[0]

In [28]:
for seq in range(len(missingPositions)):
    print(missingPositions[seq])
    alleles = list(set(missingAlleles[seq]))
    pol1 = oneTree.map_mutations(genotypes=missingGenotypes[seq], alleles = missingAlleles[seq], ancestral_state = alleles[0])
    pol2 = oneTree.map_mutations(genotypes=missingGenotypes[seq], alleles = missingAlleles[seq], ancestral_state = alleles[1])

39885
39895
39912
39989
40060
40113
40159
40212
40239
40242
40257
40269
40270
40283


In [29]:

if len(pol1[1]) > len(pol2[1]):
    mappedAncestral = pol2[0]
elif len(pol2[1]) > len(pol1[1]):
    mappedAncestral = pol1[0]
elif len(pol1[1]) == len(pol2[1]):
    mappedAncestral = "0"

In [30]:
site_id = tables.sites.add_row(missingPositions[seq], ancestral_state=pol1[0])
parent_offset = len(tables.mutations)

In [31]:
parent_offset

310520

In [34]:
len(tables.mutations)

310520

In [35]:
info = f"Site {site_id}: parsimony sets ancestral state to {pol2[0]}"

for mut in pol2[1]:
    print(mut)
    parent = mut.parent
    if parent != tskit.NULL:
        parent += parent_offset
    mut_id = tables.mutations.add_row(
        site_id, node=mut.node, parent=parent, derived_state=mut.derived_state)
    info += f", and places mutation {mut_id} to {mut.derived_state} above node {mut.node}"

Mutation(id=-1, site=-1, node=4796, derived_state='G', parent=-1, metadata=b'', time=nan, edge=-1)
Mutation(id=-1, site=-1, node=3156, derived_state='A', parent=0, metadata=b'', time=nan, edge=-1)


In [36]:
len(tables.mutations)

310522

In [ ]:
if refTmp == ancestral:
    af = sum(genoTmp) / len(genoTmp)
else:
    af = sum(1 - genoTmp) / len(genoTmp)

In [ ]:
af

0.990210843373494